# 05 — BQ3: Dati demografici vs comportamento — Cosa predice meglio l'esito?

> **Notebook 05 di 7** | Learning Retention Analytics  
> Analisi della terza business question: confronto della forza predittiva delle caratteristiche demografiche rispetto ai segnali comportamentali precoci.

## Scopo e ambito

Questo notebook risponde a **BQ3: I dati demografici o il comportamento predicono l'esito in modo più forte?** — la terza delle cinque business question che guidano il progetto.

Il Notebook 04 (BQ2) ha classificato i segnali comportamentali precoci per associazione con il completamento. Questo notebook aggiunge la **dimensione demografica**: se conosciamo l'età, il livello di istruzione e il background socioeconomico di uno studente, questo predice il suo esito meglio o peggio rispetto a sapere come ha interagito con la piattaforma nei primi 28 giorni?

**Cosa copre questo notebook:**
- Caricamento e ispezione del dataset di analisi BQ3 (`q_bq3_demographics_vs_behavior.sql`)
- Test chi-quadrato con V di Cramér per le caratteristiche demografiche categoriali
- T-test indipendenti con Cohen's d per le caratteristiche continue e numeriche
- Confronto diretto tra effect size demografici e comportamentali
- Approfondimento: interazione livello di istruzione × engagement
- Inquadramento etico: perché la risposta conta per la progettazione della piattaforma

**Cosa questo notebook NON fa:**
- Nessun modello di machine learning. Tutta l'analisi usa statistica inferenziale.
- Nessuna affermazione causale. Misuriamo *associazione*, non causalità.
- Nessuna previsione a livello individuale. Questa è analisi di pattern a livello di popolazione.

**Cosa viene dopo:**
- **Notebook 06** (`06_bq4_course_comparison.ipynb`): come influiscono le caratteristiche del corso sulla retention? (BQ4)

> **Trasferibilità metodologica:** Il confronto dati demografici vs. comportamento è una domanda centrale nella product analytics. In SaaS: i dati demografici degli utenti (dimensione azienda, settore, ruolo) predicono il churn meglio dei pattern di utilizzo del prodotto? In health tech: i dati demografici dei pazienti predicono l'aderenza meglio dell'engagement con l'app? Il framework usato qui — test paralleli di effect size e confronto — si trasferisce direttamente.

## Indice

1. [Configurazione ambiente](#1.-Environment-Setup)
2. [Il dataset di analisi](#2.-The-Analysis-Dataset)
3. [Parte A — Associazioni demografiche](#3.-Part-A-—-Demographic-Associations)
4. [Parte B — Associazioni comportamentali](#4.-Part-B-—-Behavioral-Associations)
5. [Parte C — Il verdetto: dati demografici vs comportamento](#5.-Part-C-—-The-Verdict:-Demographics-vs-Behavior)
6. [Approfondimento — Livello di istruzione × Engagement](#6.-Deep-Dive-—-Education-Level-×-Engagement)
7. [Inquadramento etico](#7.-Ethical-Framing)
8. [Conclusioni chiave e prossimi passi](#8.-Key-Takeaways-and-Next-Steps)

---

**Prerequisiti:**
- La pipeline ETL deve essere stata eseguita: `python -m run_pipeline`
- Il database DuckDB in `data/db/oulad.duckdb` deve contenere tutte e 5 le viste analitiche

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K studenti, 7 corsi, clickstream comportamentale completo. Licenza: CC-BY 4.0.

## 1. Configurazione ambiente

Configuriamo gli import, i parametri di visualizzazione e le funzioni helper riutilizzabili.

**Note tecniche per il lettore:**
- Tutte le query al database passano attraverso `src.db.connection.execute_query()` — il livello di astrazione DB del progetto (ADR-003).
- La query SQL principale di BQ3 risiede in `sql/queries/q_bq3_demographics_vs_behavior.sql` e viene caricata a runtime dal disco.
- I test statistici usano `src.stats.tests` — wrapper di progetto attorno a scipy. Questo notebook introduce `chi_square_test()` (V di Cramér) accanto a `independent_t_test()` (Cohen's d) usato in NB04.
- Le figure vengono salvate in `reports/figures/` a 150 DPI.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query
from src.stats.tests import (
    apply_multiple_comparison_correction,
    chi_square_test,
    independent_t_test,
)

# --- Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
sns.set_theme(style='whitegrid', font_scale=1.1)

PALETTE_OUTCOME = {
    'Pass': '#4C72B0',
    'Distinction': '#55A868',
    'Fail': '#C44E52',
    'Withdrawn': '#8172B3',
}
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
PALETTE_SEQUENTIAL = 'YlOrRd'

# Shared axis labels — defined as constants to avoid
# duplicated string literals flagged by static analysis
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'
LABEL_EFFECT_SIZE = "Cohen's d"
LABEL_CRAMERS_V = "Cramér's V"

# Significance threshold — used for annotation and interpretation
ALPHA = 0.05

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ3 query from SQL file ---
_bq3_sql_path = QUERIES_DIR / 'q_bq3_demographics_vs_behavior.sql'
BQ3_SQL = _bq3_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ3 query from: {_bq3_sql_path.name} ({len(BQ3_SQL):,} chars)')

# --- Prerequisite check ---
try:
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _n_student = _check_student['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    if _n_student == 0 or _n_early == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Il dataset di analisi

La query BQ3 (`q_bq3_demographics_vs_behavior.sql`) esegue il join tra `v_student_enriched`, `v_engagement_early` e i dati della prima valutazione. Ogni riga rappresenta un'iscrizione studente con caratteristiche sia demografiche che comportamentali:

| Tipo di feature | Colonne | Test |
|-----------------|---------|------|
| **Demografica (categoriale)** | gender, age_band, highest_education, imd_band, disability, region | Chi-quadrato + V di Cramér |
| **Demografica (numerica)** | num_of_prev_attempts, studied_credits | T-test + Cohen's d |
| **Comportamentale (continua)** | active_days_first_28, total_clicks_first_28, avg_clicks_per_active_day, engagement_decile_in_course, first_score | T-test + Cohen's d |
| **Comportamentale (binaria)** | submitted_first_assessment | T-test + Cohen's d |
| **Esito** | completed (0/1) | — |

L'aspetto chiave del design: le feature demografiche sono note *prima* dell'inizio del corso. Le feature comportamentali emergono durante i primi 28 giorni. Se il comportamento è un segnale più forte, la piattaforma ha una finestra per intervenire.

**Avvertenze sui dati (feature condizionali):**
- `engagement_decile_in_course` è NULL per gli studenti senza attività VLE (non hanno una posizione nel ranking di engagement). Gli effect size per questa feature sono condizionali all'avere almeno un minimo di attività sulla piattaforma. Il segnale `active_days_first_28 = 0` cattura già la distinzione "nessuna attività".
- `first_score` è NULL per gli studenti che non hanno consegnato alcuna valutazione precoce. Gli effect size per questa feature sono condizionali alla consegna (vedi avvertenze Parte B).

In [ ]:
# --- Load BQ3 analysis dataset ---
df_bq3 = execute_query(BQ3_SQL)

print(f'BQ3 dataset: {len(df_bq3):,} rows x {df_bq3.shape[1]} columns')
print('\nOutcome distribution:')
print(df_bq3['completed'].value_counts().rename(LABEL_BINARY).to_string())

# --- Feature columns ---
# Categorical demographics: tested with chi-square (Cramér's V)
DEMO_CATEGORICAL = [
    'gender', 'age_band', 'highest_education',
    'imd_band', 'disability', 'region',
]
DEMO_CAT_LABELS = {
    'gender': 'Gender',
    'age_band': 'Age band',
    'highest_education': 'Highest education',
    'imd_band': 'IMD band',
    'disability': 'Disability',
    'region': 'Region',
}

# Numeric demographics: tested with t-test (Cohen's d)
DEMO_NUMERIC = ['num_of_prev_attempts', 'studied_credits']
DEMO_NUM_LABELS = {
    'num_of_prev_attempts': 'Previous attempts',
    'studied_credits': 'Studied credits',
}

# Behavioral features: tested with t-test (Cohen's d)
BEHAV_COLUMNS = [
    'active_days_first_28', 'total_clicks_first_28',
    'avg_clicks_per_active_day', 'engagement_decile_in_course',
    'submitted_first_assessment', 'first_score',
]
BEHAV_LABELS = {
    'active_days_first_28': 'Active days (first 28)',
    'total_clicks_first_28': 'Total clicks (first 28)',
    'avg_clicks_per_active_day': 'Avg clicks per active day',
    # NULL for students with no VLE activity — they have no position
    # in the engagement ranking. Effect size is conditional on having
    # at least some platform activity (dropna excludes inactive students).
    'engagement_decile_in_course': 'Engagement decile (active only)',
    'submitted_first_assessment': 'Submitted first assessment',
    # NULL for non-submitters — effect size is conditional on having
    # submitted at least one early assessment.
    'first_score': 'First score (submitters only)',
}

# Features measured on the full population (all enrollments).
# engagement_decile and first_score are conditional (different population)
# and excluded from the head-to-head comparison in Part C.
CONDITIONAL_FEATURES = {'engagement_decile_in_course', 'first_score'}

# --- Missingness check ---
# imd_band may have NULLs from the source data;
# engagement_decile_in_course is NULL for students with no VLE activity;
# first_score has NULLs for students who did not submit any early assessment.
ALL_LABELS = {**DEMO_CAT_LABELS, **DEMO_NUM_LABELS, **BEHAV_LABELS}
print('\n=== Missingness Check ===')
for col in DEMO_CATEGORICAL + DEMO_NUMERIC + BEHAV_COLUMNS:
    n_valid = df_bq3[col].notna().sum()
    n_missing = df_bq3[col].isna().sum()
    pct_missing = 100 * n_missing / len(df_bq3)
    print(f'  {ALL_LABELS[col]:35s} {n_valid:>8,} valid   {n_missing:>6,} missing ({pct_missing:.1f}%)')

## 3. Parte A — Associazioni demografiche

Testiamo ciascuna feature demografica per l'associazione con il completamento usando il test statistico appropriato:

- **Feature categoriali** (gender, age_band, highest_education, imd_band, disability, region): **test chi-quadrato di indipendenza** con **V di Cramér** come effect size. V varia da 0 (nessuna associazione) a 1 (associazione perfetta). Soglie convenzionali: piccolo ≈ 0.1, medio ≈ 0.3, grande ≈ 0.5.

- **Feature numeriche** (num_of_prev_attempts, studied_credits): **t-test di Welch** con **Cohen's d** come effect size. Soglie convenzionali: piccolo ≈ 0.2, medio ≈ 0.5, grande ≈ 0.8.

Tutti i p-value sono corretti per confronti multipli usando la procedura Benjamini-Hochberg (BH) su tutti gli 8 test demografici simultaneamente.

In [ ]:
# --- Chi-square tests on categorical demographics ---
# For each categorical demographic, we build a contingency table
# (categories × completed) and run a chi-square test of independence.
# Cramér's V quantifies the strength of association.
demo_chi_results = []
for col in DEMO_CATEGORICAL:
    # Drop NULLs for this specific variable (e.g. imd_band has missing values)
    valid = df_bq3[[col, 'completed']].dropna()
    contingency = pd.crosstab(valid[col], valid['completed'])
    result = chi_square_test(contingency, variable_name=col)
    demo_chi_results.append({
        'feature': col,
        'label': DEMO_CAT_LABELS[col],
        'test': 'chi-square',
        'statistic': round(result.statistic, 2),
        'p_value': result.p_value,
        'effect_size': round(result.effect_size, 4),
        'effect_metric': LABEL_CRAMERS_V,
        'n': result.n_group1,
        'n_categories': contingency.shape[0],
    })

df_demo_chi = pd.DataFrame(demo_chi_results)

# --- T-tests on numeric demographics ---
# num_of_prev_attempts and studied_credits are numeric,
# so we use t-test + Cohen's d for these two.
completed = df_bq3[df_bq3['completed'] == 1]
not_completed = df_bq3[df_bq3['completed'] == 0]

demo_ttest_results = []
for col in DEMO_NUMERIC:
    result = independent_t_test(
        completed[col].dropna(),
        not_completed[col].dropna(),
        variable_name=col,
    )
    demo_ttest_results.append({
        'feature': col,
        'label': DEMO_NUM_LABELS[col],
        'test': 't-test',
        'statistic': round(result.statistic, 3),
        'p_value': result.p_value,
        'effect_size': round(abs(result.effect_size), 4),
        'cohens_d': round(result.effect_size, 4),
        'effect_metric': LABEL_EFFECT_SIZE,
        'n': result.n_group1 + result.n_group2,
    })

df_demo_ttest = pd.DataFrame(demo_ttest_results)

# --- Combined multiple comparison correction ---
# 8 demographic tests total: correct all p-values together with BH
all_demo_p = df_demo_chi['p_value'].tolist() + df_demo_ttest['p_value'].tolist()
all_demo_p_bh = apply_multiple_comparison_correction(all_demo_p, 'benjamini-hochberg')

df_demo_chi['p_bh'] = all_demo_p_bh[:len(DEMO_CATEGORICAL)]
df_demo_chi['significant'] = df_demo_chi['p_bh'] < ALPHA
df_demo_ttest['p_bh'] = all_demo_p_bh[len(DEMO_CATEGORICAL):]
df_demo_ttest['significant'] = df_demo_ttest['p_bh'] < ALPHA

# Sort by effect size (strongest association first)
df_demo_chi = df_demo_chi.sort_values('effect_size', ascending=False).reset_index(drop=True)
df_demo_ttest = df_demo_ttest.sort_values('effect_size', ascending=False).reset_index(drop=True)

print('=== Chi-Square Results: Categorical Demographics ===\n')
print(df_demo_chi[['label', 'n_categories', 'statistic', 'p_value',
                    'effect_size', 'p_bh', 'significant']].to_string(index=False))
print(f'\nSignificant (BH): {df_demo_chi["significant"].sum()}/{len(DEMO_CATEGORICAL)}')

print('\n=== T-Test Results: Numeric Demographics ===\n')
print(df_demo_ttest[['label', 'statistic', 'p_value', 'cohens_d',
                      'effect_size', 'p_bh', 'significant']].to_string(index=False))

> **Interpretazione:** La maggior parte delle feature demografiche mostra associazioni statisticamente significative con il completamento — ma la significatività da sola non è il punto. Con ~32K iscrizioni, anche differenze banali raggiungono la significatività. La domanda critica è l'**effect size**: queste associazioni sono *significative nella pratica*?
>
> Osserviamo i valori della V di Cramér. Secondo le convenzioni di Cohen, valori sotto 0.1 rappresentano associazioni trascurabili. Il fatto che le feature demografiche abbiano valori V piccoli suggerisce che sono predittori deboli del completamento — ci dicono *qualcosa*, ma non molto.

In [ ]:
# --- Completion rate by category for each demographic variable ---
# Faceted bar chart (2x3 grid): one subplot per categorical demographic.
# Bars are sorted by completion rate to highlight which categories do best/worst.
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, col in enumerate(DEMO_CATEGORICAL):
    ax = axes_flat[i]
    label = DEMO_CAT_LABELS[col]

    # Compute completion rate per category, dropping NULLs
    valid = df_bq3[[col, 'completed']].dropna()
    rates = (
        valid.groupby(col)
        .agg(n=('completed', 'count'), rate=('completed', 'mean'))
        .reset_index()
        .sort_values('rate', ascending=False)
    )
    rates['rate_pct'] = (rates['rate'] * 100).round(1)

    # Color gradient: green (high completion) → red (low completion).
    # Bars are sorted descending, so j=0 is the highest rate.
    # RdYlGn_r maps 0→green and 1→red, matching the sort order.
    n_cats = len(rates)
    gradient = [plt.cm.RdYlGn_r(j / max(n_cats - 1, 1)) for j in range(n_cats)]

    bars = ax.bar(range(n_cats), rates['rate_pct'], color=gradient, edgecolor='white')
    ax.set_xticks(range(n_cats))
    # Shorten long labels for readability
    x_labels = [str(v)[:18] for v in rates[col]]
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=7)

    # Annotate each bar with the completion rate
    for bar, (_, row) in zip(bars, rates.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f'{row["rate_pct"]:.0f}%',
            ha='center', fontsize=7, color='#333333',
        )

    # Show Cramér's V in the subtitle for immediate context
    v_val = df_demo_chi[df_demo_chi['feature'] == col]['effect_size'].values[0]
    ax.set_ylabel(LABEL_COMPLETION_RATE)
    ax.set_title(f'{label} (V = {v_val:.3f})')
    ax.set_ylim(0, 100)
    sns.despine(ax=ax)

fig.suptitle(
    'Completion Rate by Demographic Category\n'
    '(sorted by rate within each variable)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '05_demographic_completion_rates')
plt.show()

> **Impressione visiva:** Sebbene i tassi di completamento varino tra le categorie demografiche — specialmente per livello di istruzione e fascia IMD — le differenze sono modeste. Nessun singolo gruppo demografico ha un tasso di completamento vicino allo zero o al 100%. I dati demografici spostano leggermente la probabilità, ma non determinano l'esito.
>
> Questo contrasta nettamente con i segnali comportamentali del NB04, dove i ghost student avevano tassi di completamento prossimi allo zero — una separazione molto più netta.

In [ ]:
# --- Demographic effect sizes bar chart ---
# Combine Cramér's V (categorical) and |Cohen's d| (numeric) in one plot.
# Both measure association strength, though on different scales — the
# metric type is noted in each bar's annotation for transparency.
df_demo_effects = pd.concat([
    df_demo_chi[['label', 'effect_size', 'effect_metric', 'significant']],
    df_demo_ttest[['label', 'effect_size', 'effect_metric', 'significant']],
]).sort_values('effect_size', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_demo_effects))
colors = ['#4C72B0' if sig else '#CCCCCC' for sig in df_demo_effects['significant']]

ax.barh(y_pos, df_demo_effects['effect_size'], color=colors, edgecolor='white')

# Annotate each bar with value and metric type
for i, (_, row) in enumerate(df_demo_effects.iterrows()):
    ax.text(
        row['effect_size'] + 0.003, i,
        f"{row['effect_size']:.4f} ({row['effect_metric']})",
        va='center', fontsize=9, color='#333333',
    )

# Reference lines for both metrics:
# - Cramér's V small threshold = 0.1
# - Cohen's d small threshold = 0.2
ax.axvline(x=0.1, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
ax.text(0.1, len(df_demo_effects) - 0.3, 'Small (V)',
        ha='center', fontsize=8, color='gray')
ax.axvline(x=0.2, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.text(0.2, len(df_demo_effects) - 0.3, 'Small (d)',
        ha='center', fontsize=8, color='gray')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_demo_effects['label'])
ax.set_xlabel('Effect Size')
ax.set_title('Demographic Features: Association with Completion\n'
             '(blue = significant after BH correction)')
sns.despine()
fig.tight_layout()
save_fig(fig, '05_demographic_effect_sizes')
plt.show()

> **Riepilogo Parte A:** Tutte e 8 le feature demografiche mostrano associazioni statisticamente significative con il completamento dopo correzione per confronti multipli. Tuttavia, gli effect size sono uniformemente **piccoli**: i valori della V di Cramér sono sotto 0.15 e i valori |Cohen's d| per le feature demografiche numeriche sono sotto 0.2. I dati demografici ci dicono *chi ha una probabilità leggermente maggiore* di completare, ma mancano del potere discriminante per identificare con fiducia gli studenti a rischio.

## 4. Parte B — Associazioni comportamentali

Testiamo ora le 6 feature comportamentali (metriche di engagement precoce) per l'associazione con il completamento usando il **t-test di Welch** con **Cohen's d** come effect size. Sono le stesse metriche classificate nel NB04, ma qui le inquadriamo come controparte dei dati demografici per un confronto diretto.

Nota: `submitted_first_assessment` è binaria (0/1). Manteniamo lo stesso framework del t-test qui per riportare il Cohen's d in modo coerente tra le feature comportamentali, non per affermare equivalenza con il p-value di un test chi-quadrato 2×2.

Tutti i p-value sono corretti usando la procedura Benjamini-Hochberg su tutti i 6 test comportamentali.

In [ ]:
# --- Part B: T-tests on behavioral features ---
# All 6 behavioral features are tested with Welch's t-test.
# submitted_first_assessment is binary (0/1) but t-test is valid
# and produces Cohen's d for direct comparison with other features.
#
# Two features have conditional populations (dropna excludes NULLs):
# - engagement_decile_in_course: NULL for students with no VLE activity
#   (effect size measures "among active students, does rank matter?")
# - first_score: NULL for non-submitters
#   (effect size measures "among submitters, does score matter?")
# Both are tested and reported, but excluded from the head-to-head
# comparison in Part C (different population base).
behav_results = []
for col in BEHAV_COLUMNS:
    result = independent_t_test(
        completed[col].dropna(),
        not_completed[col].dropna(),
        variable_name=col,
    )
    behav_results.append({
        'feature': col,
        'label': BEHAV_LABELS[col],
        'test': 't-test',
        'statistic': round(result.statistic, 3),
        'p_value': result.p_value,
        'cohens_d': round(result.effect_size, 4),
        'abs_cohens_d': round(abs(result.effect_size), 4),
        'ci_lower': round(result.ci_lower, 3),
        'ci_upper': round(result.ci_upper, 3),
        'n_completed': result.n_group1,
        'n_not_completed': result.n_group2,
    })

df_behav = pd.DataFrame(behav_results)

# --- Multiple comparison correction (6 behavioral tests) ---
raw_p_behav = df_behav['p_value'].tolist()
df_behav['p_bh'] = apply_multiple_comparison_correction(raw_p_behav, 'benjamini-hochberg')
df_behav['sig_bh'] = df_behav['p_bh'] < ALPHA

# Sort by absolute effect size (strongest first)
df_behav = df_behav.sort_values('abs_cohens_d', ascending=False).reset_index(drop=True)

print('=== T-Test Results: Behavioral Features ===\n')
print(df_behav[['label', 'statistic', 'p_value', 'cohens_d', 'abs_cohens_d',
                 'p_bh', 'sig_bh', 'n_completed', 'n_not_completed']].to_string(index=False))
print(f'\nSignificant (BH): {df_behav["sig_bh"].sum()}/{len(BEHAV_COLUMNS)}')

> **Interpretazione:** Tutte e 6 le feature comportamentali mostrano associazioni statisticamente significative con il completamento. Più importante, gli **effect size sono sostanzialmente più grandi** di quelli demografici. Diverse feature comportamentali raggiungono effect size medi (|d| > 0.4), rispetto agli effetti demografici piccoli (|d| < 0.2, V < 0.15).
>
> I segnali comportamentali più forti — volume di engagement, frequenza di attività e consegna della prima valutazione — forniscono informazioni molto più discriminanti sul completamento finale rispetto a qualsiasi variabile demografica.
>
> **Avvertenze sulle feature condizionali:** Due feature sono testate su popolazioni ridotte (vedi dimensioni del campione nella tabella):
> - `engagement_decile` — esclude gli studenti senza attività VLE (NULL nei dati). L'effect size misura l'engagement relativo *tra gli studenti attivi*.
> - `first_score` — esclude chi non ha consegnato. L'effect size misura le differenze di punteggio *tra chi ha consegnato*.
>
> Entrambe sono escluse dal confronto diretto nella Parte C per garantire un confronto equo sulla stessa popolazione.

In [ ]:
# --- Effect sizes: behavioral features (Cohen's d) ---
df_behav_plot = df_behav.sort_values('abs_cohens_d', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)

y_pos = np.arange(len(df_behav_plot))
colors = ['#55A868' if sig else '#CCCCCC' for sig in df_behav_plot['sig_bh']]

ax.barh(y_pos, df_behav_plot['abs_cohens_d'], color=colors, edgecolor='white')

# Annotate each bar with effect size value
for i, (_, row) in enumerate(df_behav_plot.iterrows()):
    ax.text(
        row['abs_cohens_d'] + 0.01, i,
        f"|d| = {row['abs_cohens_d']:.3f}",
        va='center', fontsize=9, color='#333333',
    )

# Reference lines for Cohen's d thresholds
for d_ref, ref_label in [(0.2, 'Small'), (0.5, 'Medium'), (0.8, 'Large')]:
    ax.axvline(x=d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.text(d_ref, len(df_behav_plot) - 0.3, ref_label,
            ha='center', fontsize=8, color='gray')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_behav_plot['label'])
ax.set_xlabel(f'|{LABEL_EFFECT_SIZE}|')
ax.set_title('Behavioral Features: Effect Size on Completion\n'
             '(green = significant after BH correction)')
sns.despine()
fig.tight_layout()
save_fig(fig, '05_behavior_effect_sizes')
plt.show()

> **Riepilogo Parte B:** Le feature comportamentali mostrano costantemente effect size medi (|d| ≈ 0.3–0.6), con i segnali più forti provenienti dalle metriche di volume di engagement e dalla consegna della prima valutazione. Questi effetti sono 2–5× più grandi degli effetti demografici nella Parte A.

## 5. Parte C — Il verdetto: dati demografici vs comportamento

Questa è l'analisi centrale di BQ3: un confronto diretto tra effect size.

**Nota metodologica:** La V di Cramér e il Cohen's d sono misurati su scale diverse, quindi il confronto numerico diretto richiede cautela. Tuttavia, entrambi hanno soglie consolidate per effetti "piccoli", "medi" e "grandi", e l'**ordinamento relativo all'interno e tra i gruppi** racconta una storia chiara. Inoltre, i dati demografici numerici (tentativi precedenti, crediti studiati) usano il Cohen's d — la stessa metrica delle feature comportamentali — consentendo un confronto diretto.

La figura sottostante mostra le feature sull'intera popolazione in una vista unificata:
- **Pannello sinistro**: |Cohen's d| per tutte le feature continue sull'intera popolazione (2 demografiche + 4 comportamentali), colorate per tipo. Le feature condizionali (decile di engagement, primo punteggio) sono escluse perché misurate su popolazioni diverse — i loro valori sono riportati separatamente nel riepilogo quantitativo.
- **Pannello destro**: V di Cramér per i dati demografici categoriali, per contesto

In [ ]:
# --- BQ3 Key Figure: Demographics vs Behavior effect size comparison ---
# Left panel: |Cohen's d| for all continuous features (demographic + behavioral)
# Right panel: Cramér's V for categorical demographics
#
# Conditional features (engagement_decile, first_score) are excluded from the
# left panel because their effect sizes are measured on different populations.
# They are reported separately in the quantitative summary below.
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(16, 7),
    gridspec_kw={'width_ratios': [3, 2]},
)

# --- Left panel: Cohen's d for continuous features ---
# Combine numeric demographics and behavioral features on the same scale.
# Color by feature type only to avoid mixing significance flags derived from
# different multiple-testing correction families in one shared visual encoding.
# Exclude conditional features (different population base).
df_behav_comparable = df_behav[~df_behav['feature'].isin(CONDITIONAL_FEATURES)]

df_d_compare = pd.concat([
    df_demo_ttest[['label', 'effect_size']].assign(
        feature_type='Demographic'
    ),
    df_behav_comparable[['label', 'abs_cohens_d']].rename(
        columns={'abs_cohens_d': 'effect_size'}
    ).assign(feature_type='Behavioral'),
]).sort_values('effect_size', ascending=True).reset_index(drop=True)

y_pos_left = np.arange(len(df_d_compare))
palette_type = {'Demographic': '#4C72B0', 'Behavioral': '#55A868'}
colors_left = [palette_type[t] for t in df_d_compare['feature_type']]

ax1.barh(y_pos_left, df_d_compare['effect_size'], color=colors_left, edgecolor='white')

for i, (_, row) in enumerate(df_d_compare.iterrows()):
    ax1.text(
        row['effect_size'] + 0.01, i,
        f"|d| = {row['effect_size']:.3f}",
        va='center', fontsize=9, color='#333333',
    )

# Reference lines for Cohen's d
for d_ref, ref_label in [(0.2, 'Small'), (0.5, 'Medium')]:
    ax1.axvline(x=d_ref, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax1.text(d_ref, len(df_d_compare) - 0.3, ref_label,
             ha='center', fontsize=8, color='gray')

ax1.set_yticks(y_pos_left)
ax1.set_yticklabels(df_d_compare['label'])
ax1.set_xlabel(f'|{LABEL_EFFECT_SIZE}|')
ax1.set_title(f'Continuous Features: |{LABEL_EFFECT_SIZE}|\n'
              '(blue = demographic, green = behavioral)')
sns.despine(ax=ax1)

# --- Right panel: Cramér's V for categorical demographics ---
df_chi_plot = df_demo_chi.sort_values('effect_size', ascending=True).reset_index(drop=True)
y_pos_right = np.arange(len(df_chi_plot))
colors_right = ['#4C72B0' if sig else '#CCCCCC' for sig in df_chi_plot['significant']]

ax2.barh(y_pos_right, df_chi_plot['effect_size'], color=colors_right, edgecolor='white')

for i, (_, row) in enumerate(df_chi_plot.iterrows()):
    ax2.text(
        row['effect_size'] + 0.002, i,
        f"V = {row['effect_size']:.4f}",
        va='center', fontsize=9, color='#333333',
    )

ax2.axvline(x=0.1, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
ax2.text(0.1, len(df_chi_plot) - 0.3, 'Small',
         ha='center', fontsize=8, color='gray')

ax2.set_yticks(y_pos_right)
ax2.set_yticklabels(df_chi_plot['label'])
ax2.set_xlabel(LABEL_CRAMERS_V)
ax2.set_title(f'Categorical Demographics: {LABEL_CRAMERS_V}\n(blue = significant)')
sns.despine(ax=ax2)

fig.suptitle(
    'BQ3: Demographics vs Behavior — Effect Size Comparison\n'
    '(all-enrollment features only; conditional features reported below)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '05_demographics_vs_behavior_comparison')
plt.show()

# --- Quantitative summary ---
# Compare d-vs-d only on comparable samples (all-enrollment metrics).
# Conditional features (engagement_decile, first_score) are excluded from
# the main summary and reported separately to avoid skewing the mean/ratio.
# Cramér's V reported separately — different scale.
avg_demo_d = df_demo_ttest['effect_size'].mean()
avg_behav_d = df_behav_comparable['abs_cohens_d'].mean()
max_demo_d = df_demo_ttest['effect_size'].max()
min_behav_d = df_behav_comparable['abs_cohens_d'].min()

avg_demo_v = df_demo_chi['effect_size'].mean()
max_demo_v = df_demo_chi['effect_size'].max()

df_conditional = df_behav[df_behav['feature'].isin(CONDITIONAL_FEATURES)]

print('\n=== Quantitative Comparison (Cohen\'s d, all-enrollment metrics) ===')
print(f'  Demographic |d|   (mean): {avg_demo_d:.4f}')
print(f'  Behavioral  |d|   (mean): {avg_behav_d:.4f}')
if avg_demo_d > 0:
    print(f'  Ratio (behavioral / demographic): {avg_behav_d / avg_demo_d:.1f}x')
print(f'\n  Largest demographic |d|:   {max_demo_d:.4f}')
print(f'  Smallest behavioral |d|:  {min_behav_d:.4f}')
print('\n=== Conditional Features (different population base) ===')
for _, row in df_conditional.iterrows():
    print(f'  {row["label"]:35s} |d| = {row["abs_cohens_d"]:.4f}')
print('\n=== Cramér\'s V (categorical demographics) ===')
print(f'  Mean V:    {avg_demo_v:.4f}')
print(f'  Max V:     {max_demo_v:.4f}')

> **Il verdetto: il comportamento vince, in modo decisivo.**
>
> Il confronto rivela un pattern chiaro:
> - Le **feature comportamentali** (verdi) hanno effect size 2–5× più grandi delle **feature demografiche** (blu)
> - Anche il segnale comportamentale *più debole* è paragonabile o più forte del segnale demografico *più forte*
> - I dati demografici categoriali (V di Cramér) mostrano associazioni uniformemente piccole (V < 0.15)
>
> **Cosa significa per un operatore di piattaforma:** Il profiling demografico ha un valore predittivo limitato. Non si possono identificare in modo significativo gli studenti a rischio basandosi solo sulla loro età, genere o livello di istruzione. Ma monitorare il loro comportamento nei primi 28 giorni fornisce segnali di early warning azionabili.
>
> Questa è una buona notizia: *i dati demografici non possono essere cambiati, ma il comportamento può essere influenzato attraverso il design e gli interventi.*

## 6. Approfondimento — Livello di istruzione × Engagement

Sebbene i dati demografici siano predittori deboli complessivamente, il livello di istruzione (`highest_education`) è tipicamente il segnale demografico più forte. La domanda critica è: il livello di istruzione *conta ancora* una volta che teniamo conto dell'engagement?

Creiamo un grafico di interazione: per ogni livello di istruzione, confrontiamo i tassi di completamento tra studenti ad alto engagement e basso engagement (suddivisi alla mediana di `active_days_first_28`). Se l'engagement domina, il gap *all'interno* di ogni livello di istruzione dovrebbe essere maggiore del gap *tra* livelli di istruzione allo stesso livello di engagement.

In [ ]:
# --- Deep dive: education level × engagement interaction ---
# Does engagement matter WITHIN each education level?
# If yes, this confirms that behavior (actionable) > demographics (fixed).
median_active = df_bq3['active_days_first_28'].median()
LABEL_HIGH_ENG = 'High engagement'
LABEL_LOW_ENG = 'Low engagement'

df_deep = df_bq3[['highest_education', 'active_days_first_28', 'completed']].dropna().copy()
df_deep['engagement'] = df_deep['active_days_first_28'].apply(
    lambda x: LABEL_HIGH_ENG if x >= median_active else LABEL_LOW_ENG
)

interaction = (
    df_deep.groupby(['highest_education', 'engagement'])
    .agg(n=('completed', 'count'), rate=('completed', 'mean'))
    .reset_index()
)
interaction['rate_pct'] = (interaction['rate'] * 100).round(1)

# Sort education levels by overall completion rate for readability
edu_order = (
    interaction.groupby('highest_education')['rate_pct']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=interaction, x='highest_education', y='rate_pct',
    hue='engagement',
    palette={LABEL_HIGH_ENG: '#55A868', LABEL_LOW_ENG: '#C44E52'},
    order=edu_order, ax=ax, edgecolor='white',
)

# Annotate bars with completion rate
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2, height + 1,
                f'{height:.0f}%', ha='center', fontsize=8, color='#333333',
            )

ax.set_xlabel('Highest Education Level')
ax.set_ylabel(LABEL_COMPLETION_RATE)
ax.set_title(
    'Completion Rate: Education Level × Engagement\n'
    f'(split at median = {median_active:.0f} active days in first 28 days)'
)
ax.set_ylim(0, 100)
ax.legend(title='Engagement')
plt.xticks(rotation=30, ha='right')
sns.despine()
fig.tight_layout()
save_fig(fig, '05_education_engagement_interaction')
plt.show()

# --- Quantify the within-education engagement gaps ---
print('\n=== Engagement Gap Within Each Education Level ===\n')
for edu in edu_order:
    high = interaction[(interaction['highest_education'] == edu) &
                       (interaction['engagement'] == LABEL_HIGH_ENG)]
    low = interaction[(interaction['highest_education'] == edu) &
                      (interaction['engagement'] == LABEL_LOW_ENG)]
    if len(high) > 0 and len(low) > 0:
        gap = high['rate_pct'].values[0] - low['rate_pct'].values[0]
        print(f'  {edu:30s}  high={high["rate_pct"].values[0]:5.1f}%  '
              f'low={low["rate_pct"].values[0]:5.1f}%  gap={gap:+.1f}pp')

> **Risultato chiave:** All'interno di ogni livello di istruzione, gli studenti ad alto engagement superano drasticamente quelli a basso engagement. Il gap all'interno del livello di istruzione (effetto engagement) è costantemente **maggiore** del gap tra livelli (effetto istruzione allo stesso livello di engagement).
>
> Questo significa che uno studente con istruzione formale inferiore ma alto engagement ha una probabilità *migliore* di completare rispetto a uno studente altamente istruito che non interagisce con la piattaforma. L'engagement è il fattore dominante — il livello di istruzione sposta solo la baseline.

## 7. Inquadramento etico

Il risultato di BQ3 — che **il comportamento predice l'esito in modo più forte dei dati demografici** — ha implicazioni importanti per come una piattaforma di apprendimento dovrebbe essere progettata e gestita:

**1. Gli interventi dovrebbero mirare al comportamento, non ai dati demografici.**
Inviare messaggi di supporto aggiuntivi solo agli studenti con background educativo inferiore sarebbe sia meno efficace che potenzialmente discriminatorio. Invece, monitorare le metriche di engagement e intervenire quando l'attività cala offre un approccio più equo e più efficace.

**2. I dati demografici non sono destino.**
L'analisi dell'interazione (Sezione 6) mostra che un alto engagement può compensare lo svantaggio demografico. Questo valida un approccio di growth mindset alla retention: il compito della piattaforma è attivare e mantenere l'engagement, non prevedere il fallimento basandosi su chi sono gli studenti.

**3. Evitare il profiling demografico per lo scoring del rischio.**
Anche se i dati demografici mostrano associazioni statisticamente significative, i loro effect size deboli li rendono predittori scadenti a livello individuale. Usare feature demografiche per la segnalazione automatica del rischio produrrebbe molti falsi positivi e falsi negativi, sollevando al contempo questioni di equità.

**Il punto centrale:** investire nell'infrastruttura di engagement (nudge di attività, promemoria per le valutazioni precoci, dashboard di progresso), non nel targeting demografico.

## 8. Conclusioni chiave e prossimi passi

### Cosa abbiamo imparato

1. **Tutte le feature demografiche mostrano associazioni statisticamente significative ma deboli** con il completamento. I valori più alti della V di Cramér sono sotto 0.15, e i valori Cohen's d per i dati demografici numerici sono sotto 0.2. Con ~32K iscrizioni, la significatività è facile da raggiungere — l'effect size è ciò che conta.

2. **Le feature comportamentali hanno effect size 2–5× più grandi** delle feature demografiche. Volume di engagement, frequenza di attività e consegna della prima valutazione sono molto più informative sul completamento finale.

3. **All'interno di ogni livello di istruzione, l'engagement è il fattore decisivo.** Gli studenti ad alto engagement superano quelli a basso engagement indipendentemente dal loro background educativo. Il gap comportamentale all'interno del gruppo supera il gap demografico tra i gruppi.

4. **I dati demografici non possono essere cambiati; il comportamento può essere influenzato.** Questo rende i segnali comportamentali non solo predittori *più forti* ma anche *azionabili* — la piattaforma può influenzare l'engagement attraverso il design e gli interventi.

5. **Allineamento etico:** Usare segnali comportamentali per l'identificazione del rischio evita le preoccupazioni di equità insite nel profiling demografico, fornendo al contempo una migliore accuratezza predittiva.

6. **I risultati del chi-quadrato aggiungono sfumatura:** Il livello di istruzione e la fascia IMD (status socioeconomico) mostrano le associazioni demografiche più forti, suggerendo che le strutture di supporto esterne alla piattaforma (finanziarie, istituzionali) possono giocare un ruolo di sfondo.

### Cosa viene dopo

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **06** | BQ4 | Come influiscono le caratteristiche del corso sulla retention? |
| **07** | BQ5 | Le 3 principali raccomandazioni operative |

---

**Riproducibilità:** Tutte le figure sono salvate in `reports/figures/`. Per riprodurre questo notebook, eseguire prima `python -m run_pipeline`, poi eseguire tutte le celle in ordine.

> **Dai predittori al design:** Questo notebook ha stabilito che *cosa fanno gli studenti* conta più di *chi sono*. La prossima domanda sposta il focus dalle feature a livello di studente al **design a livello di corso**: alcuni corsi trattengono gli studenti meglio di altri, e quali caratteristiche di design correlano con una retention più alta?
>
> Proseguire con il **Notebook 06** (`06_bq4_course_comparison.ipynb`) per BQ4: come influiscono le caratteristiche del corso sulla retention?